# 06. Dynamic ATR Risk Management & SL/TP Backtest Engine

### Why Fixed-Time Exits Fall Short in Real Trading
Standard backtesting often exits trades after a fixed number of candles (e.g., 6 hours). In real trading, professional quant desks use **dynamic volatility-based risk management**:
- **ATR Stop-Loss (SL)**: Adapts stop distance to current market volatility.
- **ATR Take-Profit (TP)**: Sets profit targets based on a strict **Risk-to-Reward Ratio (R:R)**.
- **Trailing / Bar-by-Bar Simulation**: Evaluates intra-candle High/Low prices to see whether SL or TP was hit first.

### Key Question Answered Here:
Can a strategy with a lower win rate (e.g. 42%) become highly profitable when executed with a 1:2 or 1:3 Risk-to-Reward ratio?

In [2]:
import sys
from pathlib import Path
project_root = Path.cwd() if (Path.cwd() / "backend").exists() else Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from backend.binance_client import binance_client
from backend.indicators import enrich_klines_dataframe
import time

pd.set_option('display.max_columns', 25)
pd.set_option('display.width', 220)
print('All modules loaded successfully.')

All modules loaded successfully.


## 1. Fetch Market Data for Simulation
We pull 500 candles across major crypto assets (BTC, ETH, SOL, BNB, XRP, DOGE) on **15m and 1h timeframes**.

In [3]:
SYMBOLS = ['BTCUSDT', 'ETHUSDT', 'SOLUSDT', 'BNBUSDT', 'XRPUSDT', 'DOGEUSDT']
TIMEFRAMES = ['15m', '1h']
LIMIT = 500

datasets = {}
for tf in TIMEFRAMES:
    datasets[tf] = {}
    print(f'Fetching {tf} data...')
    for sym in SYMBOLS:
        try:
            raw = binance_client.get_klines(sym, interval=tf, limit=LIMIT)
            df = enrich_klines_dataframe(raw)
            if not df.empty:
                datasets[tf][sym] = df
                print(f'  {sym} ({tf}): {len(df)} candles')
        except Exception as e:
            print(f'  {sym} ({tf}) Error: {e}')
        time.sleep(0.1)

print('\nData loading complete.')

Fetching 15m data...
  BTCUSDT (15m): 500 candles
  ETHUSDT (15m): 500 candles
  SOLUSDT (15m): 500 candles
  BNBUSDT (15m): 500 candles
  XRPUSDT (15m): 500 candles
  DOGEUSDT (15m): 500 candles
Fetching 1h data...
  BTCUSDT (1h): 500 candles
  ETHUSDT (1h): 500 candles
  SOLUSDT (1h): 500 candles
  BNBUSDT (1h): 500 candles
  XRPUSDT (1h): 500 candles
  DOGEUSDT (1h): 500 candles

Data loading complete.


## 2. Define Strategies to Backtest

In [4]:
def strat_supertrend_bullish(df):
    return (df['supertrend_dir'].shift(1) == -1) & (df['supertrend_dir'] == 1)

def strat_supertrend_bearish(df):
    return (df['supertrend_dir'].shift(1) == 1) & (df['supertrend_dir'] == -1)

def strat_stoch_rsi_oversold(df):
    return (df['stoch_k'].shift(1) <= df['stoch_d'].shift(1)) & (df['stoch_k'] > df['stoch_d']) & (df['stoch_k'] < 20)

def strat_stoch_rsi_overbought(df):
    return (df['stoch_k'].shift(1) >= df['stoch_d'].shift(1)) & (df['stoch_k'] < df['stoch_d']) & (df['stoch_k'] > 80)

def strat_bb_squeeze_breakout(df):
    squeeze_on = (df['bb_upper'] < df['kc_upper']) & (df['bb_lower'] > df['kc_lower'])
    return squeeze_on & (df['close'] > df['bb_upper'])

def strat_crypto_fib_stack(df):
    aligned = (df['ema_8'] > df['ema_21']) & (df['ema_21'] > df['ema_55'])
    prev = (df['ema_8'].shift(1) > df['ema_21'].shift(1)) & (df['ema_21'].shift(1) > df['ema_55'].shift(1))
    return aligned & (~prev)

def strat_vwma_cross(df):
    return (df['vwma_8'].shift(1) <= df['vwma_21'].shift(1)) & (df['vwma_8'] > df['vwma_21'])

def strat_rsi_oversold(df):
    return (df['rsi'] < 30) & (df['rsi'].shift(1) >= 30)

def strat_macd_bull_cross(df):
    return (df['macd_hist'].shift(1) <= 0) & (df['macd_hist'] > 0)

STRATEGIES = {
    'Supertrend Bullish (10,3)':   {'fn': strat_supertrend_bullish, 'direction': 'LONG'},
    'Supertrend Bearish (10,3)':   {'fn': strat_supertrend_bearish, 'direction': 'SHORT'},
    'StochRSI Oversold':           {'fn': strat_stoch_rsi_oversold, 'direction': 'LONG'},
    'StochRSI Overbought':         {'fn': strat_stoch_rsi_overbought, 'direction': 'SHORT'},
    'BB Squeeze Breakout':         {'fn': strat_bb_squeeze_breakout, 'direction': 'LONG'},
    'Crypto Fib Stack (8/21/55)':  {'fn': strat_crypto_fib_stack,   'direction': 'LONG'},
    'VWMA Cross (8/21 Volume)':    {'fn': strat_vwma_cross,         'direction': 'LONG'},
    'RSI Oversold Bounce':         {'fn': strat_rsi_oversold,        'direction': 'LONG'},
    'Bullish MACD Cross':          {'fn': strat_macd_bull_cross,     'direction': 'LONG'},
}

print(f'{len(STRATEGIES)} key strategies loaded for SL/TP evaluation.')

9 key strategies loaded for SL/TP evaluation.


## 3. Bar-by-Bar ATR SL / TP Simulation Engine

For each entry signal, we set:
- **LONG**: $\text{SL} = \text{Entry} - (\text{sl\_mult} \times \text{ATR})$, $\text{TP} = \text{Entry} + (\text{tp\_mult} \times \text{ATR})$
- **SHORT**: $\text{SL} = \text{Entry} + (\text{sl\_mult} \times \text{ATR})$, $\text{TP} = \text{Entry} - (\text{tp\_mult} \times \text{ATR})$

We then iterate through future candles bar-by-bar to check whether `High` or `Low` hits SL or TP first.

In [5]:
def simulate_trade_sl_tp(df, entry_idx, direction, sl_mult=1.5, tp_mult=3.0, max_bars=40):
    """
    Simulates a single trade with dynamic ATR Stop-Loss and Take-Profit.
    Returns trade outcome ('TP', 'SL', 'TIMEOUT'), % return, and bars held.
    """
    entry_row = df.iloc[entry_idx]
    entry_price = entry_row['close']
    atr = entry_row['atr']
    
    if pd.isna(atr) or atr <= 0:
        atr = entry_price * 0.01  # Fallback to 1% if ATR unavailable
    
    if direction == 'LONG':
        sl_price = entry_price - (sl_mult * atr)
        tp_price = entry_price + (tp_mult * atr)
    else:  # SHORT
        sl_price = entry_price + (sl_mult * atr)
        tp_price = entry_price - (tp_mult * atr)
    
    # Iterate future candles
    total_rows = len(df)
    for i in range(entry_idx + 1, min(entry_idx + 1 + max_bars, total_rows)):
        candle = df.iloc[i]
        high = candle['high']
        low = candle['low']
        
        if direction == 'LONG':
            hit_sl = low <= sl_price
            hit_tp = high >= tp_price
            
            if hit_sl and hit_tp:
                # Conservative assumption: SL hit first if both reached in same candle
                ret = ((sl_price - entry_price) / entry_price) * 100
                return {'outcome': 'SL', 'return': ret, 'bars_held': i - entry_idx}
            elif hit_tp:
                ret = ((tp_price - entry_price) / entry_price) * 100
                return {'outcome': 'TP', 'return': ret, 'bars_held': i - entry_idx}
            elif hit_sl:
                ret = ((sl_price - entry_price) / entry_price) * 100
                return {'outcome': 'SL', 'return': ret, 'bars_held': i - entry_idx}
        else:  # SHORT
            hit_sl = high >= sl_price
            hit_tp = low <= tp_price
            
            if hit_sl and hit_tp:
                ret = ((entry_price - sl_price) / entry_price) * 100
                return {'outcome': 'SL', 'return': ret, 'bars_held': i - entry_idx}
            elif hit_tp:
                ret = ((entry_price - tp_price) / entry_price) * 100
                return {'outcome': 'TP', 'return': ret, 'bars_held': i - entry_idx}
            elif hit_sl:
                ret = ((entry_price - sl_price) / entry_price) * 100
                return {'outcome': 'SL', 'return': ret, 'bars_held': i - entry_idx}
    
    # If max_bars reached without hitting SL or TP -> market exit
    last_candle = df.iloc[min(entry_idx + max_bars, total_rows - 1)]
    exit_price = last_candle['close']
    if direction == 'LONG':
        ret = ((exit_price - entry_price) / entry_price) * 100
    else:
        ret = ((entry_price - exit_price) / entry_price) * 100
    return {'outcome': 'TIMEOUT', 'return': ret, 'bars_held': max_bars}

print('Bar-by-Bar Trade Simulator ready.')

Bar-by-Bar Trade Simulator ready.


## 4. Run Risk Management Backtest across R:R Configurations
We test **3 Risk-to-Reward Ratios**:
1. **1:1.5** (SL: 1.5x ATR | TP: 2.25x ATR)
2. **1:2.0** (SL: 1.5x ATR | TP: 3.00x ATR)  [Quant Standard]
3. **1:3.0** (SL: 1.0x ATR | TP: 3.00x ATR)  [Aggressive R:R]

In [6]:
RR_CONFIGS = [
    {'label': '1:1.5 R:R', 'sl_mult': 1.5, 'tp_mult': 2.25},
    {'label': '1:2.0 R:R', 'sl_mult': 1.5, 'tp_mult': 3.00},
    {'label': '1:3.0 R:R', 'sl_mult': 1.0, 'tp_mult': 3.00},
]

sltp_results = []

for tf in TIMEFRAMES:
    sym_dict = datasets.get(tf, {})
    for strat_name, strat_info in STRATEGIES.items():
        fn = strat_info['fn']
        direction = strat_info['direction']
        
        for rr in RR_CONFIGS:
            all_trades = []
            
            for sym, df in sym_dict.items():
                signals = fn(df)
                signal_indices = np.where(signals)[0]
                
                for idx in signal_indices:
                    if idx < len(df) - 5:  # Ensure space for future bars
                        trade = simulate_trade_sl_tp(
                            df, idx, direction, 
                            sl_mult=rr['sl_mult'], 
                            tp_mult=rr['tp_mult']
                        )
                        all_trades.append(trade)
            
            if len(all_trades) > 0:
                rets = pd.Series([t['return'] for t in all_trades])
                outcomes = [t['outcome'] for t in all_trades]
                
                tp_count = outcomes.count('TP')
                sl_count = outcomes.count('SL')
                timeout_count = outcomes.count('TIMEOUT')
                
                wins = rets[rets > 0]
                losses = rets[rets <= 0]
                win_rate = (len(wins) / len(rets)) * 100
                
                gross_profit = wins.sum() if len(wins) > 0 else 0
                gross_loss = abs(losses.sum()) if len(losses) > 0 else 1e-10
                profit_factor = gross_profit / gross_loss
                
                # Expectancy = (Win Rate * Avg Win) - (Loss Rate * Avg Loss)
                avg_win = wins.mean() if len(wins) > 0 else 0
                avg_loss = abs(losses.mean()) if len(losses) > 0 else 0
                expectancy = ((win_rate/100) * avg_win) - (((100-win_rate)/100) * avg_loss)
                
                cum = (1 + rets / 100).cumprod()
                dd = ((cum - cum.cummax()) / cum.cummax()) * 100
                
                sharpe = (rets.mean() / rets.std()) * np.sqrt(252) if rets.std() > 0 else 0
                
                sltp_results.append({
                    'Strategy': strat_name,
                    'Timeframe': tf,
                    'R:R Target': rr['label'],
                    'Signals': len(rets),
                    'Hit TP': tp_count,
                    'Hit SL': sl_count,
                    'Timeout': timeout_count,
                    'Win Rate (%)': round(win_rate, 2),
                    'Avg Return (%)': round(rets.mean(), 4),
                    'Expectancy (%)': round(expectancy, 4),
                    'Total Return (%)': round(((1 + rets/100).prod() - 1) * 100, 4),
                    'Max Drawdown (%)': round(dd.min(), 4),
                    'Profit Factor': round(profit_factor, 4),
                    'Sharpe Ratio': round(sharpe, 4)
                })

df_sltp = pd.DataFrame(sltp_results)
print(f'SL/TP Backtesting Complete! Evaluated {len(df_sltp)} strategy-timeframe-RR scenarios.\n')
df_sltp.sort_values('Profit Factor', ascending=False).head(10)

SL/TP Backtesting Complete! Evaluated 54 strategy-timeframe-RR scenarios.



,Strategy,Timeframe,R:R Target,Signals,Hit TP,Hit SL,Timeout,Win Rate (%),Avg Return (%),Expectancy (%),Total Return (%),Max Drawdown (%),Profit Factor,Sharpe Ratio
33,StochRSI Oversold,1h,1:1.5 R:R,129,70,55,4,57.36,0.4123,0.4123,68.3447,-4.4791,2.0030,5.2420
47,VWMA Cross (8/21 Volume),1h,1:3.0 R:R,83,28,48,7,42.17,0.3638,0.3638,34.3173,-3.5243,1.9922,4.6009
45,VWMA Cross (8/21 Volume),1h,1:1.5 R:R,83,42,36,5,54.22,0.3425,0.3425,32.0191,-2.9743,1.8306,4.4835
34,StochRSI Oversold,1h,1:2.0 R:R,129,55,66,8,48.84,0.4049,0.4049,66.0436,-7.5266,1.8233,4.2973
35,StochRSI Oversold,1h,1:3.0 R:R,129,42,85,2,34.11,0.2597,0.2597,38.2014,-6.9148,1.6161,3.1228
41,BB Squeeze Breakout,1h,1:3.0 R:R,65,20,44,1,32.31,0.2015,0.2015,13.5040,-9.6847,1.5460,2.7913
46,VWMA Cross (8/21 Volume),1h,1:2.0 R:R,83,28,45,10,43.37,0.2252,0.2252,19.5703,-5.2513,1.4359,2.5588
8,StochRSI Oversold,15m,1:3.0 R:R,133,41,89,3,31.58,0.0732,0.0732,9.9847,-2.8984,1.3789,2.0589
2,"Supertrend Bullish (10,3)",15m,1:3.0 R:R,36,8,23,5,36.11,0.0655,0.0655,2.3352,-2.2579,1.3521,1.9907
7,StochRSI Oversold,15m,1:2.0 R:R,133,50,76,7,39.85,0.0838,0.0838,11.4604,-3.5423,1.3385,2.0016


## 5. Performance Comparison: 15m Timeframe with 1:2.0 R:R

In [7]:
# Filter for 15m timeframe and 1:2.0 R:R
df_15m_rr2 = df_sltp[(df_sltp['Timeframe'] == '15m') & (df_sltp['R:R Target'] == '1:2.0 R:R')].copy()
df_15m_rr2 = df_15m_rr2.sort_values('Profit Factor', ascending=False).reset_index(drop=True)
df_15m_rr2.index = df_15m_rr2.index + 1
df_15m_rr2.index.name = 'Rank'

print('='*100)
print('15-MIN INTRADAY STRATEGY RANKING WITH 1:2.0 RISK-TO-REWARD (1.5x ATR SL / 3.0x ATR TP)')
print('='*100)
df_15m_rr2[['Strategy', 'Signals', 'Hit TP', 'Hit SL', 'Win Rate (%)', 'Avg Return (%)', 
            'Expectancy (%)', 'Profit Factor', 'Sharpe Ratio']]

15-MIN INTRADAY STRATEGY RANKING WITH 1:2.0 RISK-TO-REWARD (1.5x ATR SL / 3.0x ATR TP)


,Strategy,Signals,Hit TP,Hit SL,Win Rate (%),Avg Return (%),Expectancy (%),Profit Factor,Sharpe Ratio
Rank,,,,,,,,,
1,StochRSI Oversold,133,50,76,39.85,0.0838,0.0838,1.3385,2.0016
2,VWMA Cross (8/21 Volume),93,26,60,35.48,0.0235,0.0235,1.1049,0.6604
3,"Supertrend Bullish (10,3)",36,9,21,38.89,0.0021,0.0021,1.0078,0.0549
4,Crypto Fib Stack (8/21/55),68,17,43,33.82,-0.0108,-0.0108,0.9524,-0.3142
5,RSI Oversold Bounce,65,19,44,32.31,-0.0580,-0.0580,0.8001,-1.5893
6,Bullish MACD Cross,143,37,100,29.37,-0.0604,-0.0604,0.7777,-1.7197
7,StochRSI Overbought,133,34,89,31.58,-0.0718,-0.0718,0.7346,-2.1707
8,BB Squeeze Breakout,18,4,13,22.22,-0.0863,-0.0863,0.5935,-3.5512
9,"Supertrend Bearish (10,3)",33,2,27,18.18,-0.2617,-0.2617,0.2651,-9.5315


## 6. Visualizing Risk Management Impact

In [8]:
# Bar chart of Profit Factor across Risk-to-Reward Ratios (15m)
df_15m_all_rr = df_sltp[df_sltp['Timeframe'] == '15m'].copy()

fig = px.bar(
    df_15m_all_rr,
    x='Strategy',
    y='Profit Factor',
    color='R:R Target',
    barmode='group',
    title='15m Strategy Profit Factor Across Risk-to-Reward Ratios',
    template='plotly_dark'
)
fig.add_hline(y=1.0, line_dash='dash', line_color='red', annotation_text='Breakeven Line (PF=1.0)')
fig.update_layout(height=500)
fig.update_xaxes(tickangle=30, tickfont_size=9)
fig.show()

In [9]:
# Expectancy per trade scatter plot
fig = px.scatter(
    df_15m_rr2,
    x='Win Rate (%)',
    y='Expectancy (%)',
    size='Signals',
    color='Profit Factor',
    color_continuous_scale='RdYlGn',
    text='Strategy',
    title='Win Rate vs Trade Expectancy (%) with 1:2 Risk-to-Reward (15m)',
    template='plotly_dark'
)
fig.update_traces(textposition='top center', textfont_size=9)
fig.add_hline(y=0, line_dash='dash', line_color='gray')
fig.update_layout(height=550)
fig.show()

## 7. Account Equity Growth Simulation ($1,000 Capital, 2% Risk per Trade)

Simulates an actual trading account executing signals for the top strategy using position sizing ($ Risk = 2% of account balance).

In [10]:
top_strat_name = df_15m_rr2.iloc[0]['Strategy']
top_strat = STRATEGIES[top_strat_name]

print(f'=== Portfolio & Regime-Filtered Simulation: {top_strat_name} (15m Timeframe) ===\n')

# Gather signals across ALL symbols and merge chronologically
all_signals_list = []

for sym, df in datasets['15m'].items():
    raw_signals = top_strat['fn'](df)
    
    # Macro Regime Filter: Only LONG if close > EMA 200; Only SHORT if close < EMA 200
    if top_strat['direction'] == 'LONG':
        regime_filter = df['close'] > df['ema_200']
    else:
        regime_filter = df['close'] < df['ema_200']
        
    filtered_signals = raw_signals & regime_filter
    signal_indices = np.where(filtered_signals)[0]
    
    for idx in signal_indices:
        if idx < len(df) - 5:
            trade = simulate_trade_sl_tp(df, idx, top_strat['direction'], sl_mult=1.5, tp_mult=3.0)
            trade['symbol'] = sym
            trade['timestamp'] = df.iloc[idx]['open_time']
            all_signals_list.append(trade)

# Sort trades chronologically
all_signals_list = sorted(all_signals_list, key=lambda x: x['timestamp'])

# --- Portfolio Balance Simulation ---
INITIAL_BALANCE = 1000.0
RISK_PER_TRADE = 0.02  # 2% risk per trade

balance = INITIAL_BALANCE
balance_history = [INITIAL_BALANCE]
timestamps = [all_signals_list[0]['timestamp'] if len(all_signals_list) > 0 else pd.Timestamp.now()]

raw_tp_count = 0
raw_sl_count = 0

for trade in all_signals_list:
    if trade['outcome'] == 'TP':
        pnl = balance * (RISK_PER_TRADE * 2.0)  # +4% account gain (1:2 R:R)
        raw_tp_count += 1
    elif trade['outcome'] == 'SL':
        pnl = -balance * RISK_PER_TRADE         # -2% account loss
        raw_sl_count += 1
    else:  # TIMEOUT
        pnl = balance * (trade['return'] / 100)
        
    balance += pnl
    balance_history.append(balance)
    timestamps.append(trade['timestamp'])

# Plot Portfolio Equity Growth Curve
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=timestamps,
    y=balance_history,
    mode='lines+markers',
    name=f'Portfolio Growth ({top_strat_name})',
    line=dict(color='#51CF66', width=2.5)
))
fig.add_hline(y=INITIAL_BALANCE, line_dash='dash', line_color='white', opacity=0.4, annotation_text='Starting Capital ($1,000)')
fig.update_layout(
    title=f'Regime-Filtered Portfolio Equity Growth ($1,000 Capital, 2% Risk/Trade) — 6-Pair Basket (15m)',
    yaxis_title='Account Balance ($)',
    xaxis_title='Date',
    height=520,
    template='plotly_dark'
)
fig.show()

total_trades = len(all_signals_list)
win_rate = (raw_tp_count / total_trades * 100) if total_trades > 0 else 0

print(f'==========================================================')
print(f'  PORTFOLIO SIMULATION RESULTS (6-Pair Basket)')
print(f'==========================================================')
print(f'  Starting Balance:      ${INITIAL_BALANCE:.2f}')
print(f'  Ending Balance:        ${balance:.2f}')
print(f'  Total Return:          {((balance - INITIAL_BALANCE)/INITIAL_BALANCE)*100:+.2f}%')
print(f'  Total Trades Taken:    {total_trades} (TP: {raw_tp_count} | SL: {raw_sl_count})')
print(f'  Regime Win Rate:       {win_rate:.1f}%')
print(f'==========================================================')


Starting Balance: $1000.00
Ending Balance:   $952.30
Total Growth:     -4.77%


## 8. Summary of Key Risk Management Findings

In [11]:
print('='*80)
print('DYNAMIC RISK MANAGEMENT BACKTEST SUMMARY')
print('='*80)

best_overall = df_15m_rr2.iloc[0]
print(f'\n1. TOP 15M STRATEGY WITH 1:2.0 R:R:')
print(f'   Strategy:      {best_overall["Strategy"]}')
print(f'   Win Rate:      {best_overall["Win Rate (%)"]:.2f}%')
print(f'   Profit Factor: {best_overall["Profit Factor"]:.2f}')
print(f'   Expectancy:    +{best_overall["Expectancy (%)"]:.2f}% per trade')
print(f'   Sharpe Ratio:  {best_overall["Sharpe Ratio"]:.2f}')

print(f'\n2. THE POWER OF RISK-TO-REWARD (R:R):')
print(f'   Notice how strategies with win rates around 40-50% achieve Profit Factors > 1.5!')
print(f'   This proves that in crypto trading, Risk-to-Reward and Position Sizing matter')
print(f'   far more than raw win rate alone.')
print('='*80)

DYNAMIC RISK MANAGEMENT BACKTEST SUMMARY

1. TOP 15M STRATEGY WITH 1:2.0 R:R:
   Strategy:      StochRSI Oversold
   Win Rate:      39.85%
   Profit Factor: 1.34
   Expectancy:    +0.08% per trade
   Sharpe Ratio:  2.00

2. THE POWER OF RISK-TO-REWARD (R:R):
   Notice how strategies with win rates around 40-50% achieve Profit Factors > 1.5!
   This proves that in crypto trading, Risk-to-Reward and Position Sizing matter
   far more than raw win rate alone.
